# Enriquecimento dos Dados

Nesse notebook, nosso objetivo é enriquecer o nosso dataset criando duas features para `latitude` e `longitude` a partir da feature endereco.

## Importando as Bibliotecas

Vamos importar nossas bibliotecas.

In [25]:
import numpy as np
import pandas as pd

## Carregando o Dataset

Agora, vamos carregar o nosso dataset.

In [26]:
df = pd.read_parquet("../data/processed/01_cleaned.parquet")

## Enriquecendo o Dataset

Agora iremos enriquecer nosso dataset com os dados de coordenadas a partir da feature `endereco`. Primeiro vamos consultar a API do Google Maps para buscar as coordenadas dos endereços e guardar em um dicionário.

In [44]:
import os
import googlemaps

from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
gmaps = googlemaps.Client(key=API_KEY)

pattern = r"(?P<addr>.*), (?P<city>.*) - GO"
replace = lambda m: m.group("addr") + ", Goiânia - GO"

df["endereco"] = (df["endereco"]
    .str.replace(pattern, replace, regex=True)
)

addresses = df["endereco"].unique()
address_coordinates = dict()

for address in tqdm(addresses, desc="Processing Address"):
    try:
        result = gmaps.geocode(address)

        if result:
            geometry = result[0]["geometry"]["location"]
            
            latitude = geometry["lat"]
            longitude = geometry["lng"]
            
            address_coordinates[address] = (latitude, longitude)
        else:
            print(f"Nenhum resultado encontrado em '{address}'")

    except Exception as e:
        print(f"Ocorreu um erro ao consultar a API: {e}")

Processing Address:   0%|          | 0/3876 [00:00<?, ?it/s]

Processing Address: 100%|██████████| 3876/3876 [16:32<00:00,  3.91it/s]


Agora vamos guardar essas informações no `pd.DataFrame`.

In [45]:
def create_coordinates_features(reg):
    coordinates = address_coordinates[reg["endereco"]]

    reg["latitude"] = coordinates[0]
    reg["longitude"] = coordinates[1]

    return reg

df = df.apply(create_coordinates_features, axis="columns")

In [46]:
df.head()

,preco,condominio,iptu,endereco,tamanho,quartos,banheiros,vagas_estacionamento,andar,piscina,...,gramado,janelas_grandes,ventilacao_natural,fogao,trilha_caminhada,vista_panoramica,massagem,freezer,latitude,longitude
index,,,,,,,,,,,,,,,,,,,,,
0,949900,790,1900,"Rua C238, 100 - Jardim América, Goiânia - GO",130,3,4,2,24,True,...,False,False,False,False,False,False,False,False,-16.709058,-49.277839
1,760000,<NA>,<NA>,"Rua T 30, S/N - Setor Bueno, Goiânia - GO",100,3,3,2,<NA>,False,...,False,False,False,False,False,False,False,False,-16.665117,-49.233589
2,350000,270,250,"Avenida Marialva, 435 - Vila Rosa, Goiânia - GO",57,2,1,1,19,False,...,False,False,False,False,False,False,False,False,-16.742692,-49.286282
3,523000,500,800,"Rua VV 5, 1 - Residencial Eldorado, Goiânia - GO",74,3,3,1,1,True,...,False,False,False,False,False,False,False,False,-16.711875,-49.321283
4,624700,500,900,"Avenida Dona Maria Cardoso, 735 - Parque Amazô...",72,2,1,1,8,False,...,False,False,False,False,False,False,False,False,-16.737753,-49.280492


In [47]:
df = df.drop("endereco", axis="columns")

## Convertendo os `Dtypes`

Vamos converter os `dtypes` do dataset.

In [48]:
df = df.convert_dtypes()

## Salvando o Dataset

Por fim, vamos salvar o dataset.

In [49]:
df.to_parquet("../data/processed/02_enriched.parquet")